# Mirco-Benchmarking for Transformers

This notebook benchmarks the most time consuming components in BERT, GPT-2 and T5 to help you understand its performance. Let's first check our libraries and hardware. If your GPUs are recent models, please make sure your CUDA version is also recent, which may greatly affect the performance.

In [3]:
import torch

print('Pytorch version\t:', torch.__version__)
print('CUDA version\t:', torch.version.cuda)
print('GPU\t\t:',torch.cuda.get_device_name())

Pytorch version	: 2.1.1+cu121
CUDA version	: 12.1
GPU		: Tesla V100-PCIE-32GB


Let's first define a `walltime` method to benchmark Pytorch statements by at least 3 seconds. 

In [5]:
import inspect
from collections import defaultdict
import pandas as pd
from torch.utils import benchmark 

pd.options.display.precision = 3

def var_dict(*args):
    callers_local_vars = inspect.currentframe().f_back.f_locals.items()
    return dict([(name, val) for name, val in callers_local_vars if val is arg][0] 
                for arg in args)

def walltime(stmt, arg_dict, duration=3):
    return benchmark.Timer(stmt=stmt, globals=arg_dict).blocked_autorange(
        min_run_time=duration).median

Last install huggingface from source code.

In [ ]:
from IPython.display import clear_output

!git clone https://github.com/huggingface/transformers
!cd transformers; pip install .

clear_output()

## Matrix Multiplication

Matrix multiplication is the most used operator in Transformers. Its performance is crucial. Let's test the [TFLOPS](https://en.wikipedia.org/wiki/FLOPS) we can achieve on square matrices. 

In [6]:
matmul_tflops = defaultdict(lambda: {})
for n in [128, 512, 2048, 8192]:
    for dtype in (torch.float32, torch.float16):
        a = torch.randn(n, n, dtype=dtype).cuda()
        b = torch.randn(n, n, dtype=dtype).cuda()   
        t = walltime('a @ b', var_dict(a, b))
        matmul_tflops[f'n={n}'][dtype] = 2*n**3 / t / 1e12
        del a, b
        
pd.DataFrame(matmul_tflops)

,n=128,n=512,n=2048,n=8192
torch.float32,0.231,8.570,13.278,13.164
torch.float16,0.221,12.798,71.329,86.695


You can see that the performance increases with the matrix size. If your GPU has [Tensor Cores](https://www.nvidia.com/en-us/data-center/tensor-cores/), you will see a big performance jump when switching from 32-bit floating points to 16-bit floating points.

Next you can find the theory TFLOPS of your GPU from Wikipedia, for example, [Nvidia Tesla](https://en.wikipedia.org/wiki/Ampere_(microarchitecture)), [Nvidia Quadro](https://en.wikipedia.org/wiki/Quadro), [RTX 30xx](https://en.wikipedia.org/wiki/GeForce_30_series), and [RTX 20xx](https://en.wikipedia.org/wiki/GeForce_20_series). Here we list several cards, with their memory information.

| Model       | Memory (GB) | Memory Bandwidth (GB/sec) | FP32 TFLOPS | FP16 TFLOPS |
| ----------- | ----------- | ------------------------- | ----------- | ----------- |
| A100        | 80          | 2039                      | 19.5        | 312         |
| V100        | 16          | 900                       | 15.7        | 125         |
| A6000       | 48          | 768                       | 38          | 150         |
| RTX 3090 TI | 24          | 1008                      | 40          | 160         |

If the best TFLOPS number you got is still far away from the theory TFLOPS of your GPU, the performance is likely bottlenecked by the memory bandwidth. To illustrate it, let's benchmark a simple elemental-wise multiplication to show both its TFLOPS with memory bandwidth. 

In [7]:
vector = defaultdict(lambda: {})
for n in [1024*64, 1024*256, 1024*1024, 1024*1024*4]:
    a = torch.randn(n).cuda()
    t = walltime('a * 1.2', var_dict(a))
    vector[n]['TFLOPS'] = n / t / 1e12
    vector[n]['GB/s'] = 8 * n / t / 1e9
    
pd.DataFrame(vector)

,65536,262144,1048576,4194304
TFLOPS,0.004,0.015,0.061,0.095
GB/s,30.820,123.016,490.803,759.748


你可以看到，即使对于大向量，TFLOPS也远远低于GPU的峰值性能，而带宽可能非常接近其理论值。

矩阵乘法性能是高性能计算（HPC）的主要话题。有大量的研究论文。不幸的是，后端库cuBLAS并未开源。你可以查看 [cutlass](https://github.com/NVIDIA/cutlass)，它声称具有与cuBLAS相似的性能，以获取一些实现细节。

## BERT Layer

Transformer模型的主体是Transformer块的堆叠。让我们对单个块的性能进行基准测试。在BERT中，它通常被称为BERT层。让我们从[BERT大型模型](https://huggingface.co/bert-large-uncased)构建这样一层。我们使用16位浮点数以获得更好的性能。

In [10]:
from transformers import AutoConfig, BertLayer

config = AutoConfig.from_pretrained("bert-large-uncased")
layer = BertLayer(config).half().cuda()

然后定义一个函数，使用不同的序列长度和批量大小来对前向和前向反向性能进行基准测试。

In [11]:
def layer_benchmark(layer, hidden_size, seq_lens, batch_sizes, cross_attention=False):
    h = hidden_size
    results = defaultdict(lambda: {})    
    encoder_state = 'encoder_hidden_states=X' if cross_attention else ''
    for s in seq_lens:
        for b in batch_sizes:            
            ffn = 16*b*s*h*h / 1e12  # TFLOPS for the Feed-Forward Network
            atten = (4*b*h*s*s + 8*b*s*h*h) / 1e12  # TFLOPS for attention            
            forward = ffn + (2 if cross_attention else 1) * atten
            
            X = torch.randn(b, s, h).half().cuda()
            results[f'batch={b}'][f'fwd seq_len={s}'] = forward / walltime(
                f'layer(X, {encoder_state})', var_dict(layer, X))
            results[f'batch={b}'][f'fwd+bwd seq_len={s}'] = 3 * forward / walltime(
                f'layer(X, {encoder_state})[0].sum().backward()', var_dict(layer, X))            
    return pd.DataFrame(results)

在BERT预训练中，我们通常使用128（阶段1）或512（阶段2）的序列进行训练。让我们测试一下它的性能。

In [12]:
layer_benchmark(layer, config.hidden_size, [128, 512], [2, 4, 8, 16, 32, 64, 128])

,batch=2,batch=4,batch=8,batch=16,batch=32,batch=64,batch=128
fwd seq_len=128,5.766,11.578,23.676,46.295,48.601,51.879,52.670
fwd+bwd seq_len=128,5.559,4.147,8.357,48.985,33.122,60.358,62.875
fwd seq_len=512,24.695,37.814,39.572,41.637,42.426,43.119,43.445
fwd+bwd seq_len=512,8.819,17.605,35.132,48.714,50.472,51.168,51.154


毫不奇怪，大批量大小有所帮助。但最佳数字低于矩阵乘法的TFLOPS。让我们找出原因。

我们首先对层中的前馈网络（FFN）的第一层密集层进行基准测试。

In [14]:
h, b, s = config.hidden_size, 64, 128
X = torch.randn(b, s, h).half().cuda()

'Dense layer TFLOPS: %.3f' % (8*b*s*h*h / 1e12 / walltime(    
    'layer.intermediate.dense(X)', var_dict(layer, X)))

'Dense layer TFLOPS: 74.201'

数字相当好。然后，我们使用GeLU激活函数运行这个密集层。

In [15]:
'Dense+Activation TFLOPS: %.3f' % (8*b*s*h*h / 1e12 / walltime(
    'layer.intermediate(X)', var_dict(layer, X)))

'Dense+Activation TFLOPS: 63.034'

即使激活函数的复杂性可以忽略不计，它也会降低TFLOPS。我们之前指出过原因，激活函数的元素级操作受到内存带宽的限制。

现在测试整个前馈网络（FFN）。

In [19]:
ffn = 16*b*s*h*h / 1e12
'FFN TFLOPS: %.3f'%(ffn / walltime(
    'layer.output(layer.intermediate(X),X)', var_dict(layer, X)))

'FFN TFLOPS: 64.057'

BERT层中的另一部分是多头自注意力。

In [17]:
att = (4*b*h*s*s + 8*b*s*h*h) / 1e12
'Attention TFLOPS: %.3f'%(
    att / walltime('layer.attention(X)', var_dict(layer, X)))

'Attention TFLOPS: 38.161'

尽管注意力块的主要计算部分仍然是矩阵乘法，但与FFN相比，它有更多的内存绑定运算符。所以你看到的TFLOPS较低。

In [18]:
att / ffn

0.53125

注意力和FFN之间的复杂性比例取决于BERT配置。总体性能是这两个组件的FLOPS的加权和。

## GPT-2块

接下来，让我们评估`gpt2-medium`，它的架构与`bert-large`相似，即24层，隐藏大小为1024。GPT2的训练序列长度为1024。

In [21]:
from transformers.models.gpt2.modeling_gpt2 import GPT2Block

config = AutoConfig.from_pretrained("gpt2-medium")
layer = GPT2Block(config, layer_idx=0).half().cuda()
layer_benchmark(layer, config.n_embd, [512, 1024], [2, 4, 8, 16, 32, 64])

,batch=2,batch=4,batch=8,batch=16,batch=32,batch=64
fwd seq_len=512,18.954,28.761,30.742,31.589,32.238,32.534
fwd+bwd seq_len=512,8.720,17.878,32.457,35.390,36.425,36.401
fwd seq_len=1024,23.562,24.987,25.628,26.158,26.213,26.075
fwd+bwd seq_len=1024,17.501,28.217,29.667,30.365,30.366,29.882


你可以看到，尽管GPT-2和BERT具有相同的复杂性，但当使用相同的批量大小和序列长度时，GPT-2的TFLOPS稍差。此外，使用更大的序列长度1024会进一步损害性能。

## T5 layer

T5既有编码器也有解码器，让我们首先对性能与BERT类似的解码器进行基准测试。

In [24]:
from transformers.models.t5.modeling_t5 import T5Block

config = AutoConfig.from_pretrained("t5-large")
config.use_cache = False
config.is_decoder = False
config.is_encoder_decoder = False

encoder = T5Block(config).half().cuda()
layer_benchmark(encoder, config.d_model, [512], [2, 4, 8, 16, 32, 64, 128])

,batch=2,batch=4,batch=8,batch=16,batch=32,batch=64,batch=128
fwd seq_len=512,15.582,23.726,25.541,27.144,28.036,28.182,28.068
fwd+bwd seq_len=512,6.496,13.183,26.407,32.519,33.698,33.888,33.827


解码器有一个额外的交叉注意力，这增加了时间复杂性，也降低了TFLOPS。

In [25]:
config.is_decoder = True
decoder = T5Block(config).half().cuda()
layer_benchmark(decoder, config.d_model, [512], [2, 4, 8, 16, 32, 64, 128], cross_attention=True)

,batch=2,batch=4,batch=8,batch=16,batch=32,batch=64,batch=128
fwd seq_len=512,11.853,19.881,21.346,22.622,23.413,23.590,23.639
fwd+bwd seq_len=512,6.525,11.486,23.010,27.521,28.594,28.868,28.881


## 结论

总的来说，要实现Transformer层的最佳性能，你需要使用快速的数据类型和大的批量大小。要进一步提高，

我们可能需要重写代码。例如，将多个内核[fuse](https://pytorch.org/tutorials/recipes/recipes/tuning_guide.html#fuse-pointwise-operations)为一个。